# CongApp - Google Colab Virginia Legislative PDF Processing Pipeline
Use free Google Colab T4 GPU or CPU runtimes to extract text chunks, compute 384-dim FastEmbed (`BAAI/bge-small-en-v1.5`) vectors, and export JSON payload packages for CongApp Qdrant Vector Store.

In [ ]:
# Step 1: Install dependencies
!pip install -q pymupdf fastembed httpx requests

In [ ]:
# Step 2: Import libraries
import os
import json
from pathlib import Path
import pymupdf
from fastembed import TextEmbedding
import httpx

print('Colab Environment Ready!')

In [ ]:
# Step 3: Define PDF Extraction and FastEmbed Processor
def process_virginia_pdf(pdf_path: str, state: str = 'Virginia', year: int = 2020):
    doc = pymupdf.open(pdf_path)
    doc_title = Path(pdf_path).name
    chunks = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text('text')
        words = text.split()
        if not words:
            continue
        
        for i in range(0, len(words), 225):
            chunk_text = ' '.join(words[i:i+250])
            chunks.append({
                'doc_title': doc_title,
                'text_chunk': chunk_text,
                'page': page_num + 1,
                'paragraph': i // 225 + 1,
                'section': 'Virginia General Assembly Act',
                'ordinance_id': doc_title.split('.')[0],
                'date': f'{year}-01-01',
                'state': state,
                'vector': None
            })
    doc.close()
    
    print(f'Extracted {len(chunks)} text chunks. Computing FastEmbed vectors...')
    embedding_model = TextEmbedding('BAAI/bge-small-en-v1.5')
    texts = [c['text_chunk'] for c in chunks]
    vectors = list(embedding_model.embed(texts))
    
    for idx, vec in enumerate(vectors):
        chunks[idx]['vector'] = vec.tolist()
        
    return chunks


In [ ]:
# Step 4: Export to JSON or Push to Backend API
# sample_chunks = process_virginia_pdf('Virginia_HB-1526_2020.pdf', state='Virginia', year=2020)
# with open('ingested_chunks.json', 'w') as f:
#     json.dump({'chunks': sample_chunks}, f, indent=2)
# print('Export Complete: ingested_chunks.json')